# AgentLab: LLM'yi gözlemle / Observe the LLM

Bu defterde gerçek bir modelin tokenlarını, sonraki token olasılıklarını ve araç çağrısını inceleyeceğiz. Hücreleri sırayla çalıştırın. Kodları değiştirip karşılaştırın.

**İndirme:** Qwen3.5-0.8B açık model ağırlıkları yaklaşık **1,75 GB**; kütüphaneler için ek alan ve bağlantı gerekir. Colab ücretsiz GPU sağlamayabilir; CPU da desteklenir ancak yavaştır. Model ve örnekler eğitim amaçlıdır; gizli bilgi girmeyin. İlerleyerek indirmeyi başlatmış olursunuz.

Türkçe ve İngilizce yönergeler AgentLab deposundaki LLM deney alanı klasöründe bulunur.

### 0A. Ortam ve aygıt denetimi / Environment and device check
Colab veya yerel Jupyter ortamını, Python sürümünü ve varsa GPU aygıtını denetler; hiçbir indirme başlatmaz.

In [ ]:
# AMAÇ / MODÜL: 0A — Oturum 1 başlangıç ortamını (Colab/yerel Python ve GPU/CPU aygıtını) denetler.
# SINIR / NOT: Hiçbir paket kurmaz veya model indirmez; yalnız mevcut çalışma zamanını raporlar.
import platform
import sys

# Colab modülü içe aktarılabiliyorsa çalışma zamanı Google Colab'dır.
try:
    __import__("google.colab")
    print("Ortam: Google Colab")
except ImportError:
    print("Ortam: yerel Python / Jupyter")

print("Python:", sys.version.split()[0], "İşletim sistemi:", platform.system())

# PyTorch varsa CUDA GPU'nun erişilebilir olup olmadığını bildir; yoksa CPU'da devam edilir.
try:
    import torch

    print(
        "Aygıt:",
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    )
except ImportError:
    print("PyTorch henüz kurulu değil.")


## 1. Kütüphaneler / Libraries
Colab'da PyTorch genellikle hazırdır. Aşağıdaki **1A** hücresi Transformers ve görselleştirme araçlarını kurar veya günceller (tek satırlık `%pip` magic hücresi olarak ayrılmıştır). Başka bir Jupyter ortamında ayrıca PyTorch gerekebilir. Colab yeniden başlatma isterse defteri baştan çalıştırın.

In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1,<2" matplotlib ipywidgets safetensors


## 2. Model ve tokenizer / Model and tokenizer
Modelin kendi tokenizer'ı kullanılır. Ağırlıklar yalnızca bu defterin geçici klasörüne indirilir; model dosyaları depoya eklenmez. Farklı modele ait veriler gösterilmez. Bu bölüm beş odaklı adımdan (`2A`–`2E`) oluşur.

### 2A. Sabit Qwen/Qwen3.5-0.8B revizyon manifesti / Pinned model manifest
Değişken `main` dalı yerine sabit bir commit kimliği ve beklenen dosya özetleri tanımlanır.

In [ ]:
# AMAÇ / MODÜL: 2A — Sabit Qwen/Qwen3.5-0.8B revizyonunu ve beklenen dosya bütünlük özetlerini tanımlar.
# SINIR / NOT: Bu kimlikler yalnız indirilecek revizyonu sabitler; diskte henüz hiçbir dosya oluşturmaz.
import hashlib
import os
import pathlib
import stat
import tempfile

# Kaynak repo ve commit sabitlemesi (değişken main dalı yerine tek revizyon).
_AGENTLAB_REVISION = "2fc06364715b967f1860aea9cf38778875588b17"
_AGENTLAB_REPO = "models--Qwen--Qwen3.5-0.8B"

# HF cache içinde 64 haneli SHA-256 (LFS) ve 40 haneli Git blob SHA-1 özetleri beklenir.
_AGENTLAB_BLOBS = {
    ".gitattributes": "52373fe24473b1aa44333d318f578ae6bf04b49b",
    "LICENSE": "f938136e3adacfd92be087f6e113b5d6d97f678f",
    "README.md": "5824f1761b2b3a55a2141a9a1172a7f92c7c2ad9",
    "chat_template.jinja": "0ef09f214eaa6d9bca297988afc1454b5827b2c7",
    "config.json": "715f0448b9d38103211f0ad88bbb4d6e4f4be8c9",
    "merges.txt": "a494e019ca1502219fd0128658b979e5f05ae8e8",
    "model.safetensors-00001-of-00001.safetensors": "04b1c301231dd422b8860db31311ab2721511346a32cb1e079c4c4e5f1fe4696",
    "model.safetensors.index.json": "f691cefdb79d73270895ebd6d9594ddcecfc1838",
    "preprocessor_config.json": "2ea84a437d448ff71b08df68fdd949d5cc4ebb64",
    "tokenizer.json": "5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42",
    "tokenizer_config.json": "fae3ce993e07c092ad024dde45e592379fde91bb",
    "video_preprocessor_config.json": "3ba673a5ad7d4d13f54155ecd38b2a94a6dac8fe",
    "vocab.json": "0aa0ce0658d60ac4a5d609f4eadb0e8e43514176",
}
# huggingface_hub tarafından .no_exist altında tutulabilen bilinen sıfır baytlık işaretçiler.
_AGENTLAB_ABSENT = {
    "added_tokens.json", "special_tokens_map.json", "tokenizer.model",
    "model.safetensors", "pytorch_model.bin", "pytorch_model.bin.index.json",
    "generation_config.json", "adapter_config.json", "processor_config.json",
    "chat_template.json", "model.gguf",
}
_AGENTLAB_CACHE_TAG = (
    "Signature: 8a477f597d28d172789f06886806bc55\n"
    "# This file is a cache directory tag created by huggingface_hub.\n"
    "# For information about cache directory tags, see:\n"
    "#\thttps://bford.info/cachedir/\n"
)


### 2B. Dosya sistemi ve sembolik bağlantı bütünlük tarayıcısı / Filesystem integrity walker
Geçici önbellek dizinindeki dosya, klasör ve dahili sembolik bağlantıları TOCTOU değişim korumasıyla tarar.

In [ ]:
# AMAÇ / MODÜL: 2B — Geçici önbellek dizininin dosya/symlink envanterini ve SHA-256 / Git SHA-1 özetlerini çıkarır.
# SINIR / NOT: Kök dizin dışına yönelen sembolik bağlantıları ve okuma sırasındaki dosya mutasyonlarını reddeder.
# Klasör kimliği ve içerik envanteri bu notebook oturumuna aittir.
def _agentlab_entry(path, root):
    info = path.lstat()
    if stat.S_ISLNK(info.st_mode):
        target = os.readlink(path)
        # Sembolik bağlantı hedefinin asla geçici önbellek kökü dışına çıkmamasını doğrula.
        try:
            path.resolve(strict=False).relative_to(root.resolve())
        except ValueError as exc:
            raise RuntimeError("Klasör dışına yönelen bağlantı bulundu: " + str(path)) from exc
        return ("link", info.st_mode, info.st_mtime_ns, target)
    if stat.S_ISDIR(info.st_mode):
        return ("directory", info.st_mode, info.st_mtime_ns)
    if stat.S_ISREG(info.st_mode):
        digest = hashlib.sha256()
        git_digest = hashlib.sha1(("blob " + str(info.st_size) + "\0").encode())
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
                git_digest.update(block)
        # Okuma sırasında başka süreç dosyayı değiştirdiyse TOCTOU farkını yakala.
        after = path.lstat()
        if (info.st_dev, info.st_ino, info.st_size, info.st_mtime_ns) != (
            after.st_dev, after.st_ino, after.st_size, after.st_mtime_ns
        ):
            raise RuntimeError("Dosya denetim sırasında değişti: " + str(path))
        return (
            "file", info.st_mode, info.st_size, info.st_mtime_ns,
            digest.hexdigest(), git_digest.hexdigest(),
        )
    raise RuntimeError("Desteklenmeyen dosya türü: " + str(path))


def _agentlab_inventory(root):
    root = pathlib.Path(root)
    root_stat = root.lstat()
    if not stat.S_ISDIR(root_stat.st_mode) or root.is_symlink():
        raise RuntimeError("Model klasörü güvenli bir dizin değil.")
    records = {}

    def visit(directory):
        with os.scandir(directory) as entries:
            for entry in sorted(entries, key=lambda item: item.name):
                path = pathlib.Path(entry.path)
                name = path.relative_to(root).as_posix()
                records[name] = _agentlab_entry(path, root)
                if records[name][0] == "directory":
                    visit(path)

    visit(root)
    return records


### 2C. Önbellek düzeni ve değişim denetimi / Cache layout and mutation checks
İndirme sonrasında yalnızca sabit Qwen manifestiyle uyumlu blob, kilit ve snapshot bağlantılarının oluştuğunu doğrular.

In [ ]:
# AMAÇ / MODÜL: 2C — İndirilen önbelleğin yalnızca izinli Hugging Face yapılarını içerdiğini denetler.
# SINIR / NOT: Bu biçim denetimi dosya bütünlüğünü doğrular ancak dosyaların sahipliğini tek başına kanıtlamaz.
def _agentlab_verify_initial_cache(root, records):
    """İlk içeriğin cache biçimini denetle; dosya sahipliği çıkarımı yapma."""
    hub = "hf/hub"
    repo = hub + "/" + _AGENTLAB_REPO
    snap = repo + "/snapshots/" + _AGENTLAB_REVISION
    absent = repo + "/.no_exist/" + _AGENTLAB_REVISION
    locks = hub + "/.locks/" + _AGENTLAB_REPO
    directories = {
        "hf", hub, hub + "/.locks", locks, repo, repo + "/blobs",
        repo + "/snapshots", snap, repo + "/.no_exist", absent,
    }
    blob_ids = set(_AGENTLAB_BLOBS.values())
    # CACHEDIR.TAG Linux/Colab'da LF, Windows'ta CRLF satır sonu kullanabilir.
    tag_digests = {
        hashlib.sha256(_AGENTLAB_CACHE_TAG.encode()).hexdigest(),
        hashlib.sha256(_AGENTLAB_CACHE_TAG.replace("\n", "\r\n").encode()).hexdigest(),
    }
    for name, entry in records.items():
        kind = entry[0]
        if kind == "directory" and name in directories:
            continue
        if name == hub + "/CACHEDIR.TAG" and kind == "file" and entry[4] in tag_digests:
            continue
        if name.startswith(locks + "/") and kind == "file" and entry[2] == 0:
            lock_name = name[len(locks) + 1:]
            if lock_name.endswith(".lock") and lock_name[:-5] in blob_ids:
                continue
        if name.startswith(absent + "/") and kind == "file" and entry[2] == 0:
            if name[len(absent) + 1:] in _AGENTLAB_ABSENT:
                continue
        if name.startswith(repo + "/blobs/") and kind == "file":
            blob = name[len(repo + "/blobs/"):]
            digest = entry[4] if len(blob) == 64 else entry[5]
            if blob in blob_ids and digest == blob:
                continue
        if name.startswith(snap + "/") and kind == "link":
            filename = name[len(snap) + 1:]
            blob = _AGENTLAB_BLOBS.get(filename)
            if blob and (root / name).resolve() == (root / repo / "blobs" / blob).resolve():
                continue
        raise RuntimeError("İndirme sırasında bilinmeyen içerik oluştu: " + name)


def _agentlab_check_cache(state):
    # Yeniden çalıştırmada kök klasörün device/inode kimliği ve anlık envanteri değişmemiş olmalıdır.
    if not isinstance(state, dict) or state.get("snapshot") is None:
        raise RuntimeError("İndirme yarım veya envanter eksik; otomatik işlem durdu.")
    root = state["root"]
    info = root.lstat()
    if root.is_symlink() or (info.st_dev, info.st_ino) != state["identity"]:
        raise RuntimeError("Model klasörünün kimliği değişmiş; otomatik işlem durdu.")
    if _agentlab_inventory(root) != state["snapshot"]:
        raise RuntimeError("Model klasörü değişmiş; otomatik işlem durdu, dosyaları inceleyin.")


### 2D. İzole geçici önbellek ve silmeme güvenlik kapısı / Cache lifecycle & fail-closed guard
Oturum için izole geçici klasör oluşturur, indirme sonrasında envanteri mühürler ve sahiplik belirsizliğinde disk silmeyi reddeder.

In [ ]:
# AMAÇ / MODÜL: 2D — Geçici önbellek yaşam döngüsünü, ortam doğrulamasını ve silmeme güvenlik sınırını tanımlar.
# SINIR / NOT: _agentlab_remove_cache sahiplik kesinleşemediği için hiçbir dosya veya klasörü otomatik silmez.
def _agentlab_begin_cache():
    state = globals().get("_agentlab_cache_state")
    if state is not None:
        _agentlab_check_cache(state)  # Yeniden çalıştırma yabancı içeriği sahiplenmez.
        return state
    root = pathlib.Path(tempfile.mkdtemp(prefix="agentlab-qwen35-"))
    info = root.lstat()
    state = {
        "root": root,
        "identity": (info.st_dev, info.st_ino),
        "snapshot": None,
        "previous_env": {
            key: os.environ.get(key)
            for key in (
                "HF_HOME", "HF_HUB_CACHE", "HF_XET_CACHE",
                "HF_HUB_DISABLE_TELEMETRY", "HF_HUB_DISABLE_XET",
            )
        },
    }
    globals()["_agentlab_cache_state"] = state
    return state


def _agentlab_seal_cache(state):
    if state["snapshot"] is not None:
        _agentlab_check_cache(state)
        return
    info = state["root"].lstat()
    if state["root"].is_symlink() or (info.st_dev, info.st_ino) != state["identity"]:
        raise RuntimeError("Model klasörünün kimliği değişmiş; envanter kaydedilmedi.")
    records = _agentlab_inventory(state["root"])
    _agentlab_verify_initial_cache(state["root"], records)
    state["snapshot"] = records


def _agentlab_remove_cache(state):
    """Cache names and hashes establish integrity, not who created each entry."""
    root = state.get("root") if isinstance(state, dict) else None
    raise RuntimeError(
        "Önbellekteki dosya ve klasörlerin kime ait olduğu kesin doğrulanamıyor; "
        "hiçbir dosya veya klasör otomatik silinmedi. Klasörü inceleyin: " + str(root)
    )


def _agentlab_verify_hf_settings(constants, root):
    # huggingface_hub modülü daha önce başka bir önbellek yoluyla yüklendiyse sessizce devam etme.
    expected = {
        "HF_HOME": root / "hf",
        "HF_HUB_CACHE": root / "hf" / "hub",
        "HF_XET_CACHE": root / "hf" / "xet",
    }
    for name, path in expected.items():
        if pathlib.Path(getattr(constants, name)).resolve() != path.resolve():
            raise RuntimeError(
                "Hugging Face önceden farklı bir cache ile yüklenmiş; "
                "çalışma zamanını yeniden başlatıp hücreleri baştan çalıştırın."
            )
    if not constants.HF_HUB_DISABLE_TELEMETRY or not constants.HF_HUB_DISABLE_XET:
        raise RuntimeError(
            "Hugging Face önceden farklı ayarlarla yüklenmiş; "
            "çalışma zamanını yeniden başlatıp hücreleri baştan çalıştırın."
        )


### 2E. Hugging Face ortam kilidi ve model yükleme / HF environment lock & model loading
`HF_HOME` / `HF_HUB_CACHE` ortam değişkenlerini kütüphane içe aktarımlarından önce kilitleyip sabit **Qwen/Qwen3.5-0.8B** ağırlıklarını indirir ve doğrular.

In [ ]:
# AMAÇ / MODÜL: 2E — HF ortam değişkenlerini kilitler, sabit Qwen/Qwen3.5-0.8B model ve tokenizer'ı yükler.
# SINIR / NOT: Bu hücre çalıştırıldığında ~1,75 GB sabit revizyon ağırlıkları geçici klasöre indirilir.
_agentlab_cache_state = _agentlab_begin_cache()
agentlab_dir = _agentlab_cache_state["root"]

# Ortam değişkenleri huggingface_hub/transformers içe aktarılmadan önce ayarlanmalıdır.
os.environ["HF_HOME"] = str(agentlab_dir / "hf")
os.environ["HF_HUB_CACHE"] = str(agentlab_dir / "hf" / "hub")
os.environ["HF_XET_CACHE"] = str(agentlab_dir / "hf" / "xet")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch  # noqa: E402
from huggingface_hub import constants as hf_constants  # noqa: E402
from transformers import AutoModelForImageTextToText, AutoTokenizer  # noqa: E402

# İçe aktarım sonrası HF sabitlerinin bu oturum klasörüne bağlandığını doğrula.
_agentlab_verify_hf_settings(hf_constants, agentlab_dir)

MODEL = "Qwen/Qwen3.5-0.8B"
REVISION = _AGENTLAB_REVISION
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL, revision=REVISION, token=False, cache_dir=str(agentlab_dir / "hf" / "hub")
)
model = (
    AutoModelForImageTextToText.from_pretrained(
        MODEL,
        revision=REVISION,
        token=False,
        cache_dir=str(agentlab_dir / "hf" / "hub"),
        dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    .to(device)
    .eval()
)
_agentlab_seal_cache(_agentlab_cache_state)

print("Model:", MODEL, "Aygıt:", device)
context_limit = model.config.text_config.max_position_embeddings
print("Modelin teorik bağlam sınırı:", context_limit, "token; kullanılabilir bellek daha düşük olabilir.")


## 3. Tokenlara bak / Inspect tokens
Token her zaman bir kelime veya karakter değildir. Bir token tek başına çözümlendiğinde eksik Unicode baytı nedeniyle `�` görünebilir; ham parça ve ID de gösterilir. Özel tokenlar ayrıca işaretlenir.

### 3A. Token kimliği ve UTF-8 alt-kelime çözümleme fonksiyonu / Token inspection helper

In [ ]:
# AMAÇ / MODÜL: 3A — Metni özel token eklemeden parçalara ayırıp token ID tablosunu gösteren fonksiyonu tanımlar.
# SINIR / NOT: Model çıkarımı çalıştırmaz; tekil UTF-8 baytları bölündüğünde '�' oluşabileceğini görünür kılar.
import pandas as pd
from IPython.display import display


def show_tokens(text):
    # add_special_tokens=False ham kullanıcı metninin sohbet şablonu eklemeden nasıl bölündüğünü gösterir.
    ids = tokenizer.encode(text, add_special_tokens=False)
    rows = [
        {
            "ID": i,
            "ham parça": tokenizer.convert_ids_to_tokens(i),
            # Çok baytlı UTF-8 karakterler/emojiler bölünürse tek token decode çıktısı '�' gösterebilir.
            "tek başına çözülen": tokenizer.decode([i]),
            "özel": i in tokenizer.all_special_ids,
        }
        for i in ids
    ]
    print("Token sayısı:", len(ids))
    display(pd.DataFrame(rows))


### 3B. Türkçe, İngilizce ve emoji üzerinde token gözlemi / Tokenization examples
Türkçe ekler, boşluklar, emoji ve sayıların tokenlara nasıl bölündüğünü karşılaştırın.

In [ ]:
# AMAÇ / MODÜL: 3B — Türkçe, İngilizce, emoji ve sayı içeren örnek metinler üzerinde show_tokens çağrılarını çalıştırır.
# SINIR / NOT: Önceki hücrede yüklenen tokenizer gerekir; model ağırlıkları üzerinde ileri geçiş yapmaz.
show_tokens("Merhaba dünya! 👋")
show_tokens("Hello world! 👋")
show_tokens("Yapay zekâ ve 2026 agentlab")


## 4. Gerçek sonraki token olasılıkları / Real next-token probabilities
Ham olasılıklar model logits değerlerinin softmax dönüşümüdür. Sampling sonrası olasılıklar temperature, top-k ve top-p uygulandıktan sonra yeniden normalize edilir. Grafikler birbirinden farklı dağılımları gösterir; ikisini aynı şey gibi yorumlamayın.

### 4A. Temperature / Top-k / Top-p örnekleme dağılımı hesaplayıcısı / Sampling distribution calculator
Parametre tür ve aralık kontrolleri sonrasında ham dağılımı ve örnekleme sonrası dağılımı ayrı ayrı hesaplar.

In [ ]:
# AMAÇ / MODÜL: 4A — Ham logitlerden Temperature, Top-k ve Top-p örnekleme dağılımlarını güvenli şekilde hesaplar.
# SINIR / NOT: DEMO_INPUT_LIMIT (2048) eğitim sınırıdır; cihazda yeterli RAM/VRAM bulunduğunu garanti etmez.
import math

from transformers.generation.logits_process import (
    TemperatureLogitsWarper,
    TopKLogitsWarper,
    TopPLogitsWarper,
)

# Bu eğitim sınırı, cihazda yeterli RAM/GPU belleği bulunduğunu garanti etmez.
DEMO_INPUT_LIMIT = 2048


def sampling_distributions(logits, input_ids, temperature, top_k, top_p):
    # Python'da bool, int alt sınıfı olduğundan True/False değerleri açıkça reddedilir.
    if (
        isinstance(temperature, bool) or not isinstance(temperature, (int, float))
        or not 0.01 <= temperature <= 2 or not math.isfinite(temperature)
        or isinstance(top_k, bool) or not isinstance(top_k, int) or not 1 <= top_k <= 200
        or isinstance(top_p, bool) or not isinstance(top_p, (int, float))
        or not 0.01 <= top_p <= 1 or not math.isfinite(top_p)
    ):
        raise ValueError("temperature: 0.01–2, top_k: 1–200 tamsayı, top_p: 0.01–1 olmalı")
    temperature, top_p = float(temperature), float(top_p)
    if logits.ndim != 1 or not torch.isfinite(logits).all().item():
        raise ValueError("Model logits değerleri sonlu ve tek boyutlu olmalı.")
    # Ham olasılıklar ile warper zinciri sonrası yeniden normalize edilmiş dağılım ayrı tutulur.
    raw = torch.softmax(logits, dim=-1)
    scores = logits.unsqueeze(0)
    for warper in (
        TemperatureLogitsWarper(temperature),
        TopKLogitsWarper(min(top_k, logits.numel())),
        TopPLogitsWarper(top_p),
    ):
        scores = warper(input_ids, scores)
    after = torch.softmax(scores[0], dim=-1)
    if not torch.isfinite(after).all().item():
        raise ValueError("Sampling sonrası geçerli bir olasılık dağılımı oluşmadı.")
    return raw, after


### 4B. Olasılık tablosu ve görselleştirme yardımcısı / Probability table and plot helper
Yalnız son konumun logitlerini isteyerek (`logits_to_keep=1`) ham ve sampling sonrası en yüksek 8 adayı çizer.

In [ ]:
# AMAÇ / MODÜL: 4B — Verilen metin için son token logitlerinden ham vs sampling dağılım tablosu ve grafiği üretir.
# SINIR / NOT: Girdi token sayısı hem DEMO_INPUT_LIMIT hem model context_limit ile denetlenir.
import matplotlib.pyplot as plt


def next_token_distribution(text, temperature=0.8, top_k=20, top_p=0.9):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Boş olmayan metin girin.")
    inputs = tokenizer(text, return_tensors="pt").to(device)
    count = inputs.input_ids.shape[-1]
    # Uzun girdilerin modeli çalıştırmadan önce durdurulması bellek taşmasını önler.
    if count == 0 or count >= context_limit or count > DEMO_INPUT_LIMIT:
        raise ValueError("Girdi modelin veya bu deneyin güvenli girdi sınırını aşıyor.")
    # Yalnız son token konumu için logit hesaplatmak (logits_to_keep=1) VRAM/RAM kullanımını düşürür.
    with torch.inference_mode():
        logits = model(**inputs, logits_to_keep=1).logits[0, -1].float()
    raw, after = sampling_distributions(logits, inputs.input_ids, temperature, top_k, top_p)

    def table(probs):
        vals, ids = torch.topk(probs, min(8, probs.numel()))
        return pd.DataFrame(
            [
                {
                    "ID": i.item(),
                    "parça": tokenizer.convert_ids_to_tokens(i.item()),
                    "olasılık": v.item(),
                }
                for v, i in zip(vals, ids)
                if v.item() > 0
            ]
        )

    print(
        "Girdi tokenı:", count,
        "/ teorik model sınırı:", context_limit,
        "/ deney girdi sınırı:", DEMO_INPUT_LIMIT,
        "(donanım belleği daha düşük olabilir)",
    )
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, title, probs in zip(axes, ["Ham model", "Sampling sonrası"], [raw, after]):
        df = table(probs)
        display(df)
        ax.barh(df["parça"].astype(str)[::-1], df["olasılık"][::-1])
        ax.set_title(title)
        ax.set_xlabel("Olasılık")
        ax.set_xlim(0, 1)
    plt.tight_layout()
    plt.show()
    return raw, after


### 4C. Sıcaklık, top-k/top-p ve bağlam değişimi denemeleri / Parameter and context variations
Aynı girdide parametreleri değiştirin; ardından bağlamı değiştirin. Olasılıkların ve sıralamanın nasıl değiştiğini karşılaştırın.

In [ ]:
# AMAÇ / MODÜL: 4C — Varsayılan, yüksek sıcaklık/dar top-k ve farklı bağlam için next_token_distribution sonuçlarını kıyaslar.
# SINIR / NOT: Her çağrı tek ileri geçiş çalıştırır; olasılık farkları grafik ve tablolarda yan yana görünür.
raw, sampled = next_token_distribution(
    "Bugün hava", temperature=0.8, top_k=20, top_p=0.9
)
# Daha yüksek temperature ve daha dar top_k/top_p ile dağılımın nasıl değiştiğini izle.
_ = next_token_distribution("Bugün hava", temperature=1.2, top_k=10, top_p=0.8)
# Bağlam ("Kışın dağlarda") eklenince aday token sırasının kaymasını gözlemle.
_ = next_token_distribution(
    "Kışın dağlarda bugün hava", temperature=0.8, top_k=20, top_p=0.9
)


## 5. Üretim: tek token ve devamı / Generation: one token and beyond
Tek-token üretimi ile daha uzun üretimin ürettiği token sayısını karşılaştırın. Deterministik (`do_sample=False`) ve örneklemeli (`do_sample=True`) üretimi yan yana deneyin.

### 5A. Adım adım metin üretim fonksiyonu / Text generation helper

In [ ]:
# AMAÇ / MODÜL: 5A — Tek veya çok adımlı deterministik/örneklemeli metin üretim fonksiyonunu tanımlar.
# SINIR / NOT: Girdi + üretim bütçesi context_limit içinde tutulur; do_sample=False iken örnekleme argümanları verilmez.
def generate_text(
    prompt, max_new_tokens=40, temperature=0.8, top_k=20, top_p=0.9, do_sample=True
):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("Boş olmayan metin girin.")
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    if inputs.input_ids.shape[-1] > DEMO_INPUT_LIMIT:
        raise ValueError("Girdi bu deneyin sınırını aşıyor; donanım sınırı daha düşük olabilir.")
    if inputs.input_ids.shape[-1] + max_new_tokens > context_limit:
        raise ValueError("Girdi ve üretim bütçesi bağlam sınırını aşıyor")
    kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        kwargs.update(temperature=temperature, top_k=top_k, top_p=top_p)
    with torch.inference_mode():
        output = model.generate(**inputs, **kwargs)
    # Prompt tokenlarını hariç tutarak yalnız yeni üretilen token dilimini çöz.
    generated = output[0, inputs.input_ids.shape[-1] :]
    print(
        "Girdi tokenı:", inputs.input_ids.shape[-1],
        "Üretilen token:", len(generated),
        "Mod:", "sampling" if do_sample else "greedy",
    )
    print(repr(tokenizer.decode(generated, skip_special_tokens=True)))
    return generated


### 5B. Deterministik (greedy) ve örneklemeli üretim karşılaştırması / Deterministic vs sampled comparison

In [ ]:
# AMAÇ / MODÜL: 5B — Tek token, deterministik 40-token ve örneklemeli 40-token üretim çıktılarını karşılaştırır.
# SINIR / NOT: Örneklemeli (do_sample=True) koşu her çalıştırmada farklı token dizisi üretebilir.
_ = generate_text("Yapay zekâ", max_new_tokens=1, do_sample=False)
_ = generate_text("Yapay zekâ", max_new_tokens=40, do_sample=False)
_ = generate_text("Yapay zekâ", max_new_tokens=40, do_sample=True)


## 6. Araç çağrısı / Tool calling: model proposes, Python validates
Modelin yanıtı gerçek bir araç çağrısı içermeyebilir. Bu durumda bunu açıkça gösteririz. Sadece `lookup_term` ve izinli kavramlar kabul edilir; bir kez çalıştırılır ve gerçek sonuç modele geri gönderilir. Shell veya dış API çağrısı yoktur.

### 6A. İzinli ders sözlüğü ve sohbet şablonu yardımcısı / Allowed glossary and chat helper
Modelin görebileceği tek araç şemasını (`lookup_term`) ve sohbet çağrı yardımcısını tanımlar.

In [ ]:
# AMAÇ / MODÜL: 6A — İzinli TERMS sözlüğünü, lookup_term JSON şemasını ve Qwen chat şablonu yardımcısını tanımlar.
# SINIR / NOT: Hiçbir shell/ağ aracı yoktur; skip_special_tokens=False çıktıda <tool_call> sınırını görünür kılar.
TERMS = {
    "token": "Token, modelin işlediği metin parçasıdır.",
    "embedding": "Embedding, verinin sayısal vektör temsilidir.",
    "schema": "Schema, verinin beklenen yapısını tanımlar.",
}
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_term",
            "description": "Look up a short definition of a term.",
            "parameters": {
                "type": "object",
                "properties": {"term": {"type": "string", "enum": list(TERMS)}},
                "required": ["term"],
            },
        },
    }
]


def chat(messages, max_new_tokens=160):
    # Araç tanımları sohbet şablonuna eklenir; düşünme blokları eğitim demosu için kapatılır.
    prompt = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    if inputs.input_ids.shape[-1] > DEMO_INPUT_LIMIT:
        raise ValueError("Araç girdisi deneyin sınırını aşıyor.")
    if inputs.input_ids.shape[-1] + max_new_tokens > context_limit:
        raise ValueError("Araç konuşması bağlam sınırını aşıyor")
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        output[0, inputs.input_ids.shape[-1] :], skip_special_tokens=False
    )


### 6B. Katı araç çağrısı şema doğrulayıcısı / Strict tool-call parser
Model önerisinin tek ve izinli bir `lookup_term` çağrısı içerdiğini, ek etiket veya parametre enjeksiyonu içermediğini denetler.

In [ ]:
# AMAÇ / MODÜL: 6B — Model çıktısındaki tekil <tool_call> önerisini katı izinli şemaya göre ayrıştırır.
# SINIR / NOT: Çoklu çağrı, bilinmeyen araç, fazla parametre veya önek etiket enjeksiyonu durumunda None döner.
import re


def parse_allowed_call(proposed):
    """Accept one Qwen call, optionally preceded by ordinary explanatory text."""
    if not isinstance(proposed, str) or proposed.count("<tool_call>") != 1:
        return None
    prefix, call = proposed.split("<tool_call>", 1)
    # Çağrı öncesindeki düz metinde gizli XML/özel kontrol belirteci varsa reddet.
    forbidden = (
        "<tool_call", "</tool_call", "<function", "</function",
        "<parameter", "</parameter", "<|",
    )
    if any(marker in prefix for marker in forbidden):
        return None
    match = re.fullmatch(
        r"\s*<function=lookup_term>\s*<parameter=term>\s*(token|embedding|schema)\s*</parameter>\s*</function>\s*</tool_call>\s*(?:<\|(?:im_end|endoftext)\|>\s*)*",
        call,
    )
    return (prefix.strip(), match.group(1)) if match else None


### 6C. Öneri → Python doğrulaması → gözlem → nihai yanıt döngüsü / Proposal, validation & tool loop
Model önerisi Python'dan geçerse araç tek kez çalıştırılır, gözlem geri gönderilir ve ikinci bir çağrı önerilse bile çalıştırılmaz.

In [ ]:
# AMAÇ / MODÜL: 6C — Öneri -> Python şema doğrulama -> yerel sözlük gözlemi -> nihai yanıt akışını çalıştırır.
# SINIR / NOT: İkinci turda model tekrar araç çağrısı önerse bile yürütülmez; döngü tek araç adımıyla sınırlıdır.
def tool_demo(question):
    messages = [{"role": "user", "content": question}]
    proposed = chat(messages)
    print("1. Model önerisi:\n", proposed)
    parsed = parse_allowed_call(proposed)
    if parsed is None:
        print("2. Doğrulama: Geçerli, izinli tek araç çağrısı yok; araç çalıştırılmadı.")
        return
    prefix, term = parsed
    print("2. Doğrulama: lookup_term ve argüman izinli.")
    result = TERMS[term]
    print("3. Gerçek araç sonucu:", result)
    # Doğrulanmış araç çağrısını ve gözlemi konuşma geçmişine yapısal rol olarak ekle.
    messages.append(
        {
            "role": "assistant",
            "content": prefix or None,
            "tool_calls": [
                {
                    "type": "function",
                    "function": {"name": "lookup_term", "arguments": {"term": term}},
                }
            ],
        }
    )
    messages.append({"role": "tool", "content": result})
    answer = chat(messages)
    if "<tool_call>" in answer:
        print("4. Model ek araç çağrısı önerdi; bu öneri çalıştırılmadı:\n", answer)
    else:
        print("4. Modelin son yanıtı:\n", answer)


tool_demo("What is a token? Use lookup_term first.")


## 7. Belleği boşaltma / Release memory
Son hücre modeli bellekten çıkarır. Önbellek içeriğinin kime ait olduğu kesin olarak doğrulanamadığından hiçbir dosya veya klasör otomatik silinmez; hücrenin gösterdiği geçici klasörü inceleyin. / The last cell releases the model from memory. Cache files and folders are not automatically deleted because their ownership cannot be established; inspect the temporary folder shown by the cell.

In [ ]:
# AMAÇ / MODÜL: 7A — Model/tokenizer nesnelerini RAM ve CUDA VRAM'den bırakır; disk önbelleğini incelemeye bırakır.
# SINIR / NOT: _agentlab_remove_cache kasıtlı olarak RuntimeError fırlatıp hiçbir dosya veya klasörü otomatik silmez.
import gc

if "model" in globals():
    del model
if "tokenizer" in globals():
    del tokenizer
gc.collect()
if "torch" in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()

if globals().get("_agentlab_cache_state") is None:
    print("Bu oturuma ait temizlenecek AgentLab klasörü yok.")
else:
    try:
        _agentlab_remove_cache(_agentlab_cache_state)
    except (OSError, RuntimeError, KeyError, TypeError) as exc:
        raise RuntimeError(
            "Güvenli otomatik temizlik yapılamadı. Dosyaları inceleyin: " + str(exc)
        ) from exc
